# Tenacious-Bench — Day 0 Smoke Test (Colab T4)

**Week 11 · Day 0 pre-flight**

Per the challenge brief: _the Unsloth notebook completes a 5-task dummy LoRA run end to end (fp16 on T4, bf16 on L4/4090) and pushes the adapter to your HuggingFace account._

This notebook proves the training stack works **before Day 5**. If anything in this notebook fails, fix it now — Day 5 has no time to debug compute.

**What it does:**

1. Confirm a T4 (or better) GPU is attached.
2. Install Unsloth + TRL + PEFT.
3. Authenticate to HuggingFace (Colab Secrets → `HF_TOKEN`).
4. Load `unsloth/Qwen3.5-1.7B-Instruct` in 16-bit, attach LoRA.
5. Run **1 SimPO step** on 5 dummy preference pairs.
6. Push the adapter to `<your-hf-user>/tenacious-smoke-test` (private).

**Expected wall time:** 8–15 min (most of which is the first-run kernel compile).

> **QLoRA 4-bit is NOT used** — Week 11 brief mandates 16-bit LoRA.


## Step 1 — Check GPU runtime

Confirm a T4 (or better) is attached. **Runtime → Change runtime type → T4 GPU** if not.


In [2]:
import subprocess, sys

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found — NO GPU attached.")
    print(
        "⚠️  Go to Runtime → Change runtime type → T4 GPU, then reconnect and re-run."
    )
    sys.exit(0)  # stop here cleanly rather than crashing downstream

import torch

print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(
        f"bf16 supported: {torch.cuda.is_bf16_supported()}  (T4 → False fp16; L4/4090 → True bf16)"
    )
else:
    print("⚠️  CUDA not available — change runtime to T4 GPU before continuing.")

Thu Apr 30 22:44:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             28W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Clone the repo

Skip this cell if you uploaded the repo manually. Otherwise replace `<your-fork>` with your GitHub fork URL.


In [2]:
import os
REPO_URL = "https://github.com/sanoy24/tenacious-bench.git"  # ← edit me
if not os.path.isdir("/content/tenacious-bench"):
    !git clone $REPO_URL /content/tenacious-bench
%cd /content/tenacious-bench
!ls training/

Cloning into '/content/tenacious-bench'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 230 (delta 88), reused 209 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 641.70 KiB | 14.92 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/tenacious-bench
colab_smoke_test.py  requirements.txt  train_judge.py


## Step 3 — Install dependencies

First-run kernel compile takes 6–10 min on T4 — that is expected.


In [3]:
# Unsloth first — it pins compatible torch/transformers versions
!pip install -q unsloth
!pip install -q "trl>=0.9.0" "peft>=0.12.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
               sentencepiece protobuf "huggingface_hub>=0.24.0"
import trl, peft, transformers
print(f"trl={trl.__version__}  peft={peft.__version__}  transformers={transformers.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 12.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 133.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.9 MB/s eta 0:00:00
   

trl=0.24.0  peft=0.19.1  transformers=5.5.0


In [1]:
from huggingface_hub import login

login()

In [6]:
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-lrokr9bs/unsloth_15ce85dc1cd14e33864132c9235ffcf1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-lrokr9bs/unsloth_15ce85dc1cd14e33864132c9235ffcf1
  Resolved https://github.com/unslothai/unsloth.git to commit 265d16e742a33e956a7176d5ba309e7691e0da87
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


%cd /content/tenacious-bench
!python training/colab_smoke_test.py


In [ ]:
%cd /content/tenacious-bench
# !python training/colab_smoke_test.py --model unsloth/Qwen3.5-4B
!python training/colab_smoke_test.py \
  --hf-repo sanoy24/tenacious-smoke-test

/content/tenacious-bench
[smoke] Loading unsloth/Qwen3.5-4B in fp16
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#i

## Step 6 — Verify

If you see **`[smoke] ALL CHECKS PASSED`** above and the adapter is visible at `https://huggingface.co/<your-hf-user>/tenacious-smoke-test`, you are cleared to run `train.ipynb` on Day 5.
